# DASHBOARD ANA_EDU_FIN_CLI V1


## 1. Conexão

Cria as sessões Spark local e remota e o cliente DB2.


In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_sessao_spark_local import (
        GerenciadorSessaoSpark,
        ler_variavel_ambiente_local,
    )

    ambiente = ler_variavel_ambiente_local("AMBIENTE").upper()

    if ambiente != "MODELAGEM":
        ambiente = "PRODUCAO"

    gerenciador_spark = GerenciadorSessaoSpark(
        nome_sessao="etl-vinculacao-mf-insights",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "SANDBOX": "t2i2016",
            "AMBIENTE": ambiente,
        },
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        jars=[
            "/dados/shared/bin/ojdbc8.jar",
        ],
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "4g",
            "spark.serializer":
                "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "240",
            "spark.sql.sources.partitionOverwriteMode": "dynamic",
            "spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive":"true",
            "spark.sql.autoBroadcastJoinThreshold": "-1",
            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
try:
    %run ./src/utils/gerenciador_sessao_spark_remoto.ipynb
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
%%spark

import os

ambiente = ler_variavel_ambiente_spark("AMBIENTE").upper()
dominio = ler_variavel_ambiente_spark("DOMINIO").lower()
hoje = ler_variavel_ambiente_spark("HOJE")

if ambiente == "MODELAGEM":
    sandbox = ler_variavel_ambiente_spark("SANDBOX").lower()
    database = f"sbx_{sandbox}"
else:
    database = f"hive_{dominio}"

env_spark = dict(os.environ)

cliente_db2 = criar_cliente_db2_spark(env=env_spark)


## 2. Dashboard

Informe o cliente, o schema e a tabela. `None` ou texto vazio seleciona um cliente aleatório da própria tabela. O resultado é exibido abaixo da célula.


In [ ]:
from src.app import gerar_dashboard

CODIGO_CLIENTE = ""  # None ou "" seleciona um cliente aleatório.
SCHEMA = "sbx_t2i2016"
TABELA = "ana_edu_fin_cli"

resultado_dashboard = gerar_dashboard(
    codigo_cliente=CODIGO_CLIENTE,
    schema=SCHEMA,
    tabela=TABELA,
)
